# 4. Hypergeometric representation (Section 5.1) and Lemma 2

Checks against the reference:

* Eq. (*P1d*): the rim integral from Green's theorem;
* Eqs. (*ingress*) and (*interiorhyp*): the series of Gauss functions in $j$ (summed to convergence);
* **Lemma 2** (Appendix C): $|P_\gamma-P_{\gamma,J}|\le E^{(j)}_\gamma$ for several $J$.

Near first contact the overlap and the remainders are tiny, so this notebook uses the
high-precision reference `P_mp`.

In [1]:
import time
import numpy as np
import mpmath as mp
import refcheck as rc

FULL = False          # True: the larger grids (minutes to hours)
t_start = time.time()

mp.mp.dps = 25

In [2]:
rows = []
gammas = ["0.25", "0.5", "1.7"] if FULL else ["0.3", "1.7"]
for gs in gammas:
    g = mp.mpf(gs)
    for ps in ["0.05", "0.2"]:
        p = mp.mpf(ps)
        zlist = [0.5, 0.75, 1 - float(p) - 1e-6, 1 - float(p) + 1e-6, 1.0, 1 + float(p) - 1e-3]
        for zf in (zlist if FULL else [zlist[i] for i in (0, 2, 3, 5)]):
            z = mp.mpf(zf)
            ref = rc.P_mp(zf, p, g, dps=22)
            limb = z > 1 - p
            x = 1 - (z - p)**2
            J = int(mp.ceil(mp.log(mp.mpf("1e-20")) / mp.log(x))) + 5
            hyp = rc.P_ingress(z, p, g, J) if limb else rc.P_interiorhyp(z, p, g, J)
            lem = []
            for Jt in [0, 2, 5]:
                PJ = rc.P_ingress(z, p, g, Jt) if limb else rc.P_interiorhyp(z, p, g, Jt)
                lem.append(float(abs(ref - PJ) / rc.lemma2_bound(z, p, g, Jt)))
            rows.append((gs, ps, zf, "limb" if limb else "inside",
                         float(abs(rc.P_1d(z, p, g) - ref) / mp.pi), float(abs(hyp - ref) / mp.pi), max(lem)))
print(f"{'gamma':>5} {'p':>5} {'z':>9} {'region':>7} {'|P1d-ref|':>10} {'|j-series-ref|':>14} {'Lemma 2: err/bound':>19}")
for r in rows:
    print(f"{r[0]:>5} {r[1]:>5} {r[2]:9.6f} {r[3]:>7} {r[4]:10.1e} {r[5]:14.1e} {r[6]:19.6f}")
assert max(r[4] for r in rows) < 1e-20 and max(r[5] for r in rows) < 1e-18 and max(r[6] for r in rows) <= 1

gamma     p         z  region  |P1d-ref| |j-series-ref|  Lemma 2: err/bound
  0.3  0.05  0.500000  inside    8.1e-28        8.9e-23            0.718206
  0.3  0.05  0.949999  inside    5.0e-24        1.5e-27            0.340880
  0.3  0.05  0.950001    limb    1.8e-28        1.5e-27            0.986376
  0.3  0.05  1.049000    limb    4.9e-31        4.3e-31            0.999886
  0.3   0.2  0.500000  inside    2.8e-27        1.6e-22            0.321831
  0.3   0.2  0.799999  inside    5.7e-28        9.1e-24            0.290836
  0.3   0.2  0.800001    limb    9.7e-28        9.1e-24            0.914874
  0.3   0.2  1.199000    limb    1.4e-30        1.3e-30            0.999886
  1.7  0.05  0.500000  inside    3.8e-28        3.1e-23            0.667130
  1.7  0.05  0.949999  inside    8.3e-30        6.9e-29            0.276920
  1.7  0.05  0.950001    limb    9.6e-30        7.2e-29            0.988288
  1.7  0.05  1.049000    limb    5.1e-35        4.6e-35            0.999902
  1.7   0.2 

In [3]:
print(f"notebook run time: {time.time() - t_start:.0f} s")

notebook run time: 53 s
